# 🚀 Principal & Staff Level Fintech Technical Interview Mastery (Dual-Engine: Pandas & SQL)

Welcome to the **Staff / Principal Level Fintech Interview Challenge**.

All scenarios in this notebook are strictly **Advanced & Staff Level**, modeled after high-stakes quantitative analytics, risk engineering, and fraud infrastructure interviews at tier-1 institutions (**Two Sigma, Jane Street, Stripe, Brex, Block, Goldman Sachs, American Express**).

---

### ⚡ Rules of the Challenge:
- **Minimal Stakeholder Briefs:** You are given only the high-level business requirement from executive risk and quantitative trading leadership. No hints or intermediate steps are provided.
- **Dual-Engine Execution:** Solve each case in **both Pandas and pure SQL (`%%sql`)**.
- **Advanced Techniques Tested:** Multi-window time-series (`.rolling()`, `.ewm()`, `.shift()`), island-and-gap streak tracking (`(cond != cond.shift()).cumsum()`), Markov transition matrices (`pd.crosstab(normalize='index')`), multi-format parsing (`.xml`, `.dat`, `.psv`, `.tsv`, `.jsonl`, `.yaml`, `.json`), 3-way bridge joins, and vectorized underwriting engines (`np.select()`).

---
### 📁 Multi-Format Enterprise Datasets in `./data/`:
`raw_transactions.csv`, `customers.csv`, `merchants.csv`, `disputes.csv`, `fx_rates_daily.tsv`, `device_telemetry.jsonl`, `api_event_logs.json`, `kyc_audit_records.psv`, `credit_bureau_scores.xml`, `ach_clearing_settlement.dat`, `aml_sanctions_watchlist.yaml`

In [12]:
# Setup environment & in-memory SQLite relational engine with Native %%sql Execution
import sqlite3
import pandas as pd
import numpy as np
import json
import yaml
import xml.etree.ElementTree as ET
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

# 1. Initialize persistent in-memory SQLite connection
conn = sqlite3.connect(':memory:')

# 2. Universal Table Loader (supports CSV, TSV, and DataFrames)
def load_table(name, source, sep=','):
    if isinstance(source, pd.DataFrame):
        source.to_sql(name, conn, index=False, if_exists='replace')
        print(f"✅ Loaded DataFrame into table: '{name}' ({len(source)} rows)")
    elif isinstance(source, str) and os.path.exists(source):
        df = pd.read_csv(source, sep=sep)
        df.to_sql(name, conn, index=False, if_exists='replace')
        print(f"✅ Loaded '{source}' into table: '{name}' ({len(df)} rows)")

# 3. Pre-load Standard Enterprise Tables
base_dir = 'data' if os.path.exists('data') else '../data'

load_table('transactions', os.path.join(base_dir, 'raw_transactions.csv'))
load_table('customers', os.path.join(base_dir, 'customers.csv'))
load_table('merchants', os.path.join(base_dir, 'merchants.csv'))
load_table('disputes', os.path.join(base_dir, 'disputes.csv'))
load_table('fx_rates', os.path.join(base_dir, 'fx_rates_daily.tsv'), sep='\t')
load_table('telemetry', pd.read_json(os.path.join(base_dir, 'device_telemetry.jsonl'), lines=True))
load_table('bureau', pd.read_xml(os.path.join(base_dir, 'credit_bureau_scores.xml')))
# 4. Raw SQL Execution Engine
def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# 5. Register %%sql Magic & Auto-Transformer
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("\n🚀 SQL Studio Environment Active! You can now write %%sql directly in code cells.")
print("📁 Available Enterprise Files:", sorted(os.listdir(base_dir)))

✅ Loaded 'data\raw_transactions.csv' into table: 'transactions' (15000 rows)
✅ Loaded 'data\customers.csv' into table: 'customers' (4917 rows)
✅ Loaded 'data\merchants.csv' into table: 'merchants' (1897 rows)
✅ Loaded 'data\disputes.csv' into table: 'disputes' (10000 rows)
✅ Loaded 'data\fx_rates_daily.tsv' into table: 'fx_rates' (2000 rows)
✅ Loaded DataFrame into table: 'telemetry' (8000 rows)
✅ Loaded DataFrame into table: 'bureau' (4000 rows)

🚀 SQL Studio Environment Active! You can now write %%sql directly in code cells.
📁 Available Enterprise Files: ['ach_clearing_settlement.dat', 'aml_sanctions_watchlist.yaml', 'api_event_logs.json', 'credit_bureau_scores.xml', 'customers.csv', 'device_telemetry.jsonl', 'disputes.csv', 'fx_rates_daily.tsv', 'kyc_audit_records.psv', 'merchants.csv', 'raw_transactions.csv']


---
## 🏢 Case 1: High-Frequency Cross-Border Liquidity Velocity & EWMA Volatility Decay

### 📌 Scenario
> **Stakeholder:** Head of Quantitative Treasury & Liquidity Risk  
> *"Ingest `raw_transactions.csv` and daily market forex rates from `fx_rates_daily.tsv` (Tab-Separated). Convert all completed transactions to USD spot value. For each geographic region, construct a continuous daily time series computing:
> 1. A 7-day rolling total settlement volume.
> 2. A 14-day Exponentially Weighted Moving Average (EWMA, span=14) of daily transaction volume to measure momentum.
> 3. Day-over-Day volume acceleration: `(Daily Volume - 7D Rolling Avg) / 7D Rolling Avg`.
> Rank the top 2 single-day liquidity surge anomalies per region based on acceleration."*

In [13]:
# ✍️ PANDAS SOLUTION FOR CASE 1:
import pandas as pd
import numpy as np

# 📦 BLOCK 1: Ingest, Normalize Dates & Pre-Aggregate Daily FX Rates
fx=(
    pd.read_csv('data/fx_rates_daily.tsv', sep='\t')
    .assign(rate_date=lambda d: pd.to_datetime(d['rate_date'], errors='coerce').dt.normalize())
    .groupby('rate_date',as_index=False)['spot_rate']
    .mean()
)

tx=(
    pd.read_csv('data/raw_transactions.csv')
    .assign(
        tx_date=lambda d: pd.to_datetime(d['transaction_date'], errors='coerce').dt.normalize(),
        status=lambda d: d['transaction_status'].astype(str).str.strip().str.title(),
        region=lambda d: d['region'].astype(str).str.strip().str.title(),
        amount=lambda d: pd.to_numeric(d['transaction_amount'],errors='coerce').fillna(0.0)
    )
    .query("status == 'Completed'")
)

# ⛓️ BLOCK 2: Time-Series Windowing Pipeline ➔ EWMA ➔ Acceleration ➔ Top 2 Rank
top2_surges_df=(  # 1. Join & Aggregate to (region, tx_date) daily grain
    tx.merge(fx, left_on='tx_date', right_on='rate_date', how='left')
    .assign(daily_usd=lambda d: d['amount'] / d['spot_rate'].fillna(1.0))
    .groupby(['region','tx_date'], as_index=False)
    ['daily_usd']
    .sum()
    .sort_values(by=['region','tx_date'])
    .assign(  # 2. Compute Grouped Window Functions per Region
        rolling_7d_vol=lambda d: d.groupby('region')['daily_usd']
            .transform(lambda s: s.rolling(7, min_periods=1).sum()),
        rolling_7d_avg=lambda d: d.groupby('region')['daily_usd']
            .transform(lambda s: s.rolling(7, min_periods=1).mean()),
        ewma_14d_vol=lambda d: d.groupby('region')['daily_usd']
            .transform(lambda s: s.ewm(span=14).mean())
    )
    .assign(  # 3. Derive Acceleration Ratio & Rank Surges per Region
    acceleration_ratio=lambda d: ((d['daily_usd'] - d['rolling_7d_avg']) / d['rolling_7d_avg'].replace(0, np.nan)).fillna(0.0),
    regional_surge_rank=lambda d: d.groupby('region')['acceleration_ratio'].rank(method='dense',ascending=False).astype(int)
    )
    .query("regional_surge_rank <= 2")  # 4. Filter for Top 2 Surges per Region
    [['region','tx_date','daily_usd','rolling_7d_vol','ewma_14d_vol','acceleration_ratio','regional_surge_rank']]
    .sort_values(by=['region','regional_surge_rank'])
    .reset_index(drop=True)
)

# Run assertions to guarantee financial accuracy
assert len(top2_surges_df) == 8, f"Expected exactly 8 rows (2 per region), got {len(top2_surges_df)}"
assert top2_surges_df['region'].nunique() == 4, "Expected 4 distinct regions!"
assert (top2_surges_df['rolling_7d_vol'] >= top2_surges_df['rolling_7d_vol']).all(), "Rolling 7D sum cannot be smaller thatn single day volume!"
assert top2_surges_df.isna().sum().sum() == 0, "Unexpected nulls in output!"

print("Case 1 verified: All 4 regions successfully computed with exact top 2 surges!")
display(top2_surges_df)

Case 1 verified: All 4 regions successfully computed with exact top 2 surges!


,region,tx_date,daily_usd,rolling_7d_vol,ewma_14d_vol,acceleration_ratio,regional_surge_rank
0,East,2025-09-08,582.32000,806.987609,105.940041,4.051180,1
1,East,2025-09-12,1053.23000,1814.205441,232.246129,3.063823,2
2,North,2025-09-08,677.37000,908.222987,128.889786,4.220733,1
3,North,2025-09-13,1933.83000,2784.259043,369.550410,3.861908,2
4,South,2025-09-10,2741.70000,3003.419800,404.077952,5.390016,1
5,South,2025-07-01,85.07047,169.269557,34.965343,2.518018,2
6,West,2025-09-09,1596.61000,1912.182346,247.004049,4.844772,1
7,West,2025-09-12,3678.58000,5466.000098,704.549302,3.710951,2


In [14]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 1:
WITH valid_tx AS (
    -- Step 1: Clean transactions, normalize casing to Title Case, and dates to midnight
    SELECT 
        DATE(transaction_date) AS tx_date,
        UPPER(SUBSTR(TRIM(region), 1, 1)) || LOWER(SUBSTR(TRIM(region), 2)) AS region,
        CAST(transaction_amount AS REAL) AS tx_amount
    FROM transactions
    WHERE LOWER(TRIM(transaction_status)) = 'completed'
),
daily_fx AS (
    -- Step 2: Average daily forex rate (1 row per date)
    SELECT 
        DATE(rate_date) AS rate_date,
        AVG(CAST(spot_rate AS REAL)) AS spot_rate
    FROM fx_rates
    GROUP BY DATE(rate_date)
),
daily_regional_vol AS (
    -- Step 3: Join and aggregate daily volume per region
    SELECT 
        t.region,
        t.tx_date,
        SUM(t.tx_amount / COALESCE(fx.spot_rate, 1.0)) AS daily_volume_usd
    FROM valid_tx t
    LEFT JOIN daily_fx fx ON t.tx_date = fx.rate_date
    GROUP BY t.region, t.tx_date
),
window_metrics AS (
    -- Step 4: Compute 7-day rolling metrics and acceleration
    SELECT 
        region,
        tx_date,
        daily_volume_usd,
        SUM(daily_volume_usd) OVER (
            PARTITION BY region 
            ORDER BY tx_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7d_vol,
        AVG(daily_volume_usd) OVER (
            PARTITION BY region 
            ORDER BY tx_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7d_avg
    FROM daily_regional_vol
),
ranked_surges AS (
    -- Step 5: Derive acceleration ratio and rank top surges per region
    SELECT 
        region,
        tx_date,
        ROUND(daily_volume_usd, 2) AS daily_usd,
        ROUND(rolling_7d_vol, 2) AS rolling_7d_vol,
        ROUND((daily_volume_usd - rolling_7d_avg) / NULLIF(rolling_7d_avg, 0), 4) AS acceleration_ratio,
        DENSE_RANK() OVER (
            PARTITION BY region 
            ORDER BY ((daily_volume_usd - rolling_7d_avg) / NULLIF(rolling_7d_avg, 0)) DESC
        ) AS regional_surge_rank
    FROM window_metrics
)
-- Step 6: Filter Top 2 surges per region
SELECT 
    region,
    tx_date,
    daily_usd,
    rolling_7d_vol,
    acceleration_ratio,
    regional_surge_rank
FROM ranked_surges
WHERE regional_surge_rank <= 2
ORDER BY region, regional_surge_rank;


,region,tx_date,daily_usd,rolling_7d_vol,acceleration_ratio,regional_surge_rank
0,East,2025-09-08,582.32,852.55,3.7812,1
1,East,2025-09-12,1053.23,1843.40,2.9995,2
2,North,2025-09-08,677.37,888.65,4.3358,1
3,North,2025-09-12,1819.43,2648.94,3.8080,2
4,South,2025-09-10,2741.70,3615.63,4.3080,1
5,South,2025-09-09,564.03,922.29,3.2809,2
6,West,2025-09-09,1596.61,2000.14,4.5877,1
7,West,2025-09-12,3991.62,5867.00,3.7625,2


---
## 🏢 Case 2: Streaming Device Telemetry, Inter-Session Velocity & Tor Exit Contagion

### 📌 Scenario
> **Stakeholder:** Principal Security Data Scientist & Fraud Defense Lead  
> *"Parse streaming telemetry from `device_telemetry.jsonl` (JSON-Lines). For each customer, compute the inter-session latency delta between consecutive device events. Isolate high-frequency automated bot sessions where `network_type == 'Tor-Proxy'` OR `is_rooted_jailbroken == True` with an inter-session latency delta <= 60ms. Join with `customers.csv` to calculate the total compromised account balance exposure, grouped by customer `risk_tier` and `has_crypto_wallet` status, identifying high-risk crypto asset leakage."*

In [15]:
# ✍️ PANDAS SOLUTION FOR CASE 2:
import pandas as pd
import numpy as np 

# 📦 BLOCK 1: Ingest Streaming JSONL, Filter Bot Attacks & Pre-Aggregate to 1-Row Customer Grain
attacked_cust=(
    pd.read_json('data/device_telemetry.jsonl', lines=True)
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        network_type=lambda d: d['network_type'].astype(str).str.strip(),
        is_rooted=lambda d: d['is_rooted_jailbroken'].astype(bool),
        latency_ms=lambda d: pd.to_numeric(d['latency_ms'], errors='coerce').fillna(999.0)
    )
    .query("(network_type == 'Tor-Proxy' or is_rooted) and latency_ms <= 60")
    .groupby('customer_id', as_index=False)
    .agg(
        attack_sessions=('session_id', 'count'),
        avg_bot_latency=('latency_ms', 'mean')
    )
)

# ⛓️ BLOCK 2: Pipeline Chain: Clean Customers ➔ Safe 1:1 Merge ➔ GroupBy Exposure ➔ Sort
crypto_exposure_summary=(
    pd.read_csv('data/customers.csv')
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        risk_tier=lambda d: d['risk_tier'].astype(str).str.strip().str.strip(),
        has_crypto_wallet=lambda d: pd.to_numeric(d['has_crypto_wallet'], errors='coerce').fillna(0).astype(int),
        account_balance=lambda d: pd.to_numeric(d['account_balance'], errors='coerce').fillna(0.0)
    )
    .merge(attacked_cust, on='customer_id', how='inner')
    .groupby(['risk_tier','has_crypto_wallet'], as_index=False)
    .agg(
        attacked_customers_count=('customer_id', 'count'),
        total_exposed_balance_usd=('account_balance', 'sum'),
        avg_session_latency_ms=('avg_bot_latency', 'mean')
    )
    .assign(
        total_exposed_balance_usd=lambda d: d['total_exposed_balance_usd'].round(2),
        avg_session_latency_ms=lambda d: d['avg_session_latency_ms'].round(2)
    )
    .sort_values(by='total_exposed_balance_usd', ascending=False)
    .reset_index(drop=True)
)

display(crypto_exposure_summary)

# Step 4: Quality Assertions
assert len(crypto_exposure_summary) > 0, "Summary table cannot be empty!"
assert not crypto_exposure_summary.duplicated(subset=['risk_tier','has_crypto_wallet']).any(), "Duplicate grain keys found!"
assert (crypto_exposure_summary['total_exposed_balance_usd'] >= 0).all(), "Negative balance found!"
assert crypto_exposure_summary['avg_session_latency_ms'].between(0, 60.0).all(), "Latency must be <= 60ms!"
assert crypto_exposure_summary.isna().sum().sum() == 0, "Unexpected nulls in output!"

print("Case 2 verified: All sanity checks passed")

,risk_tier,has_crypto_wallet,attacked_customers_count,total_exposed_balance_usd,avg_session_latency_ms
0,Low,0,28,1161165.88,34.64
1,Medium,0,23,856900.97,35.59
2,Medium,1,15,672707.41,40.93
3,Low,1,10,411293.64,42.10
4,High,0,13,331028.77,38.23
5,Critical,0,10,301827.22,38.30
6,Critical,1,6,275839.97,33.67
7,High,1,3,158344.71,57.00


Case 2 verified: All sanity checks passed


In [16]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 2:
WITH flagged_telemetry AS (  -- Block 1: Filter bot attacks & pre-aggregate to 1 row per customer
    SELECT 
        TRIM(customer_id) AS customer_id,
        AVG(CAST(latency_ms AS REAL)) AS avg_bot_latency
    FROM telemetry
    WHERE (network_type = 'Tor-Proxy' OR is_rooted_jailbroken = 1 OR is_rooted_jailbroken = 'True')
    AND CAST(latency_ms AS REAL) <= 60
    GROUP BY TRIM(customer_id)
),

cleaned_customers AS (  -- Clean customer demographics, casing, and balances
    SELECT 
        TRIM(customer_id) AS customer_id,
        UPPER(SUBSTR(TRIM(risk_tier), 1, 1)) || LOWER(SUBSTR(TRIM(risk_tier), 2)) AS risk_tier,
        CAST(COALESCE(has_crypto_wallet, 0) AS INTEGER) AS has_crypto_wallet,
        CAST(COALESCE(account_balance, 0.0) AS REAL) AS account_balance
        FROM customers
),

joined_exposure AS (  -- 📦 BLOCK 2: Safe 1:1 Inner Join between customers and attacked accounts
    SELECT
        c.risk_tier, c.has_crypto_wallet, c.customer_id, c.account_balance, t.avg_bot_latency 
    FROM cleaned_customers c 
    INNER JOIN flagged_telemetry t 
        ON c.customer_id = t.customer_id
)

SELECT  -- Final Presentation: Group by risk_tier x has_crypto_wallet & rank by highest exposure
    risk_tier, has_crypto_wallet,
    COUNT(customer_id) AS attacked_customers_Count,
    ROUND(SUM(account_balance), 2) AS total_exposed_balance_usd,
    ROUND(AVG(avg_bot_latency), 2) AS avg_session_latency_ms
FROM joined_exposure
GROUP BY risk_tier, has_crypto_wallet
ORDER BY total_exposed_balance_usd DESC;

,risk_tier,has_crypto_wallet,attacked_customers_Count,total_exposed_balance_usd,avg_session_latency_ms
0,Low,0,28,1161165.88,34.64
1,Medium,0,23,856900.97,35.59
2,Medium,1,15,672707.41,40.93
3,Low,1,10,411293.64,42.10
4,High,0,13,331028.77,38.23
5,Critical,0,10,301827.22,38.30
6,Critical,1,6,275839.97,33.67
7,High,1,3,158344.71,57.00


---
## 🏢 Case 3: XML Credit Bureau Pulls & Multi-Dimensional Underwriting Transition Matrix

### 📌 Scenario
> **Stakeholder:** Chief Credit Risk Officer (CCRO)  
> *"Extract multi-bureau inquiry records from `credit_bureau_scores.xml`. Segment individual bureau FICO scores into standardized Underwriting Bands:
> - `Deep Subprime (<580)`
> - `Near-Prime [580, 669]`
> - `Prime [670, 739]`
> - `Super-Prime [740, 850]`
> Merge with `customers.csv`. Build a normalized cross-tabulation transition matrix comparing internal customer `risk_tier` against bureau score bands, showing both absolute customer counts and row-normalized percentage distributions, while computing total balance exposure for Subprime accounts with `hard_inquiries_12m >= 3`."*

In [17]:
# ✍️ PANDAS SOLUTION FOR CASE 3:
import pandas as pd 
import numpy as np 

# 📦 BLOCK 1: 1-Line XML Ingest & Vectorized FICO Score Binning
bureau_df=(
    pd.read_xml('data/credit_bureau_scores.xml')
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        fico_score=lambda d: pd.to_numeric(d['fico_score_8'], errors='coerce'),
        hard_inquiries_12m=lambda d: pd.to_numeric(d['hard_inquiries_12m'], errors='coerce').fillna(0),
        bureau_band=lambda d: pd.cut(d['fico_score'], bins=[300, 580, 670, 740, 851], labels=['Deep Subprime', 'Near-Prime', 'Prime', 'Super-Prime'], right=False),
    )
)

# ⛓️ BLOCK 2: Pipeline Chain: Clean Customers ➔ Safe 1:1 Merge ➔ Dual Outputs
cleaned_customers=(
    pd.read_csv('data/customers.csv')
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        risk_tier=lambda d: d['risk_tier'].astype(str).str.strip().str.title(),
        account_balance=lambda d: pd.to_numeric(d['account_balance'], errors='coerce').fillna(0.0)
    )
)

# Safe 1:1 Merge
underwriting_master=cleaned_customers.merge(bureau_df, on='customer_id', how='inner', validate='1:1')


# 📊 Deliverable 1: Credit Transition Matrix (Row % Sums to 100%)
transition_matrix=(
    pd.crosstab(
        underwriting_master['risk_tier'], underwriting_master['bureau_band'], normalize='index'
    ) * 100
).round(2)

# 💵 Deliverable 2: Distressed Subprime Balance Exposure
vulnerable_exposure_summary=(
    underwriting_master
    .query("bureau_band == 'Deep Subprime' and hard_inquiries_12m >= 3")
    .groupby('risk_tier', as_index=False)
    .agg(
        vulnerable_customers_count=('customer_id', 'nunique'),
        total_vulnerable_balance_usd=('account_balance', 'sum'),
        avg_inquiries_count=('hard_inquiries_12m', 'mean')
    )
    .round(2)
    .sort_values(by='total_vulnerable_balance_usd', ascending=False)
    .reset_index(drop=True)
)

display(transition_matrix)
display(vulnerable_exposure_summary)

# 1. Assert Transition Matrix row totals sum to 100%
np.testing.assert_allclose(transition_matrix.sum(axis=1), 100.0, rtol=1e-3)

# 2. Assert no duplicate risk tiers in exposure table
assert not vulnerable_exposure_summary.duplicated(subset=['risk_tier']).any(), "Duplicate grain found!"

# 3. Assert non-negative balance exposures
assert (vulnerable_exposure_summary['total_vulnerable_balance_usd'] >= 0).all(), "Negative balance found!"

print("Case 3 Sanity Audit Passed 100!")


bureau_band,Deep Subprime,Near-Prime,Prime,Super-Prime
risk_tier,,,,
Critical,46.44,15.79,14.09,23.68
High,47.35,18.88,13.13,20.65
Low,45.66,17.82,14.95,21.58
Medium,44.08,18.66,14.80,22.46


,risk_tier,vulnerable_customers_count,total_vulnerable_balance_usd,avg_inquiries_count
0,Low,412,16328202.66,5.50
1,Medium,389,14845055.55,5.35
2,High,211,7988921.12,5.52
3,Critical,192,7545014.16,5.28


Case 3 Sanity Audit Passed 100!


In [23]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 3:
WITH scored_bureau AS (
    SELECT 
        TRIM(customer_id) AS customer_id,
        CAST(fico_score_8 AS INTEGER) AS fico_score,
        CAST(hard_inquiries_12m AS INTEGER) AS hard_inquiries_12m,
        CASE 
            WHEN CAST(fico_score_8 AS INTEGER) < 580 THEN 'Deep Subprime'
            WHEN CAST(fico_score_8 AS INTEGER) BETWEEN 580 AND 669 THEN 'Near-Prime'
            WHEN CAST(fico_score_8 AS INTEGER) BETWEEN 670 and 739 THEN 'Prime'
            ELSE 'Super-Prime'
        END AS bureau_band
    FROM bureau
),

cleaned_customers AS (
    SELECT
        TRIM(customer_id) AS customer_id,
        UPPER(SUBSTR(TRIM(risk_tier), 1, 1)) || LOWER(SUBSTR(TRIM(risk_tier), 2)) AS risk_tier,
        CAST(COALESCE(account_balance, 0.0) AS REAL) AS account_balance
    FROM customers
),

underwriting_master AS (
    SELECT 
        c.risk_tier, c.customer_id, c.account_balance, b.fico_score,
        b.hard_inquiries_12m, b.bureau_band
    FROM cleaned_customers c 
    INNER JOIN scored_bureau b ON c.customer_id=b.customer_id     
)

-- 💵 Deliverable: Subprime Exposure Summary
SELECT 
    risk_tier,
    COUNT(DISTINCT customer_id) AS vulnerable_customers_count,
    ROUND(SUM(account_balance), 2) AS total_vulnerable_balance_usd,
    ROUND(AVG(hard_inquiries_12m), 2) AS avg_inquiries_count
FROM underwriting_master
WHERE bureau_band = 'Deep Subprime'
    AND hard_inquiries_12m >=3
GROUP BY risk_tier
ORDER BY total_vulnerable_balance_usd DESC;

,risk_tier,vulnerable_customers_count,total_vulnerable_balance_usd,avg_inquiries_count
0,Low,412,16328202.66,5.50
1,Medium,389,14845055.55,5.35
2,High,211,7988921.12,5.52
3,Critical,192,7545014.16,5.28


In [27]:
%%sql
-- 💵 DELIVERABLE 2: SUBPRIME EXPOSURE WITH HIGH INQUIRIES (>= 3)
WITH scored_bureau AS (
    SELECT 
        TRIM(customer_id) AS customer_id,
        CAST(fico_score_8 AS INTEGER) AS fico_score,
        CAST(hard_inquiries_12m AS INTEGER) AS hard_inquiries_12m,
            CASE 
                WHEN CAST(fico_score_8 AS INTEGER) < 580 THEN 'Deep Subprime'
                WHEN CAST(fico_score_8 AS INTEGER) BETWEEN 580 AND 669 THEN 'Near-Prime'
                WHEN CAST(fico_score_8 AS INTEGER) BETWEEN 670 AND 739 THEN 'PRime'
                ELSE 'Super-Prime'
            END AS bureau_band
        FROM bureau
),

cleaned_customers AS (
    SELECT 
        TRIM(customer_id) AS customer_id,
        UPPER(SUBSTR(TRIM(risk_tier), 1, 1)) || LOWER(SUBSTR(TRIM(risk_tier), 2)) AS risk_tier,
        CAST(COALESCE(account_balance, 0.0) AS REAL) AS account_balance
    FROM customers
),

underwriting_master AS (
    SELECT 
        c.risk_tier, c.customer_id, c.account_balance, b.fico_score,
        b.hard_inquiries_12m, b.bureau_band
    FROM cleaned_customers c
    INNER JOIN scored_bureau b ON c.customer_id=b.customer_id
)

SELECT 
    risk_tier,
    COUNT(DISTINCT customer_id) AS vulnerable_customers_count,
    ROUND(SUM(account_balance), 2) AS total_vulnerable_balance_usd,
    ROUND(AVG(hard_inquiries_12m), 2) AS avg_inquiries_count
FROM underwriting_master
WHERE bureau_band='Deep Subprime'
    AND hard_inquiries_12m >= 3
GROUP BY risk_tier
ORDER BY total_vulnerable_balance_usd DESC;

,risk_tier,vulnerable_customers_count,total_vulnerable_balance_usd,avg_inquiries_count
0,Low,412,16328202.66,5.50
1,Medium,389,14845055.55,5.35
2,High,211,7988921.12,5.52
3,Critical,192,7545014.16,5.28


---
## 🏢 Case 4: Acquiring Interchange Margin Arbitrage & Merchant Net Drain Portfolio

### 📌 Scenario
> **Stakeholder:** Head of Acquiring Economics & Payment Partnerships  
> *"Reconcile transaction processing revenues across `merchants.csv`, `raw_transactions.csv`, and `disputes.csv`:
> 1. Gross Processing Fee = `SUM(transaction_amount * interchange_fee_pct)`.
> 2. Operational Dispute Drain = `SUM(disputed_amount + chargeback_fee_usd)`.
> 3. Net Operating Margin = Gross Fee - Dispute Drain.
> Group by merchant `category` and isolate merchants running at a net negative operating margin (`Net Margin < 0`). Output the top 10 most unprofitable merchant accounts with their category, gross fee, dispute drain, and net loss."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 4:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 4:

---
## 🏢 Case 5: Fixed-Width NACHA Clearing Returns & Systemic Routing Corridor Risk

### 📌 Scenario
> **Stakeholder:** Director of NACHA Settlement & Clearing Compliance  
> *"Ingest the fixed-width NACHA clearing ledger `ach_clearing_settlement.dat` using column width specifications. For every bank routing number (`ROUTING_NUM`), compute the cumulative running sum of total batch settlement volume vs unauthorized returns (`RETURNED_NSF`, `RETURNED_ACT_CLOSED`, `SUSPENDED_AML`). Identify routing corridors whose cumulative unauthorized return rate exceeds 1.5% of total volume. Merge with `customers.csv` to calculate total systemic customer account balance exposure tied to breached routing corridors."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 5:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 5:

---
## 🏢 Case 6: Pipe-Separated PSV Biometric Anomaly & Geographic Spoofing Concentration

### 📌 Scenario
> **Stakeholder:** Head of Identity Intelligence & Biometric Risk  
> *"Parse regulatory verification logs from `kyc_audit_records.psv` (pipe-delimited, handling `#` header comments). Identify synthetic biometric spoofing attempts defined as: `confidence_score < 0.65` AND `facial_match_pct < 55.0` OR `screening_flags` containing `'TAMPERED_DOC_SUSPECTED'`. Cross-reference with `customers.csv` to aggregate biometric anomalies across US states (`state_province`), calculating total verification attempts, spoofing counts, and the biometric spoofing rate per state, ranked by highest risk."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 6:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 6:

---
## 🏢 Case 7: Island-and-Gap MFA Streak Anomaly & Webhook ATO Exposure

### 📌 Scenario
> **Stakeholder:** Staff Security Engineer & Incident Response Lead  
> *"Scan real-time security logs from `api_event_logs.json`. Using consecutive island-and-gap streak tracking on customer event sequences, identify accounts undergoing active credential stuffing attacks defined as: 3 or more consecutive failed MFA challenges (`(mfa_prompted == True) & (mfa_passed == False)`) OR a single event risk score >= 0.90. Merge with `customers.csv` to generate an executive report detailing total attacked accounts, average attack risk score, and total balance exposure grouped by `account_tier`."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 7:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 7:

---
## 🏢 Case 8: Hierarchical YAML Sanctions Graph Contagion & Capital Freeze Engine

### 📌 Scenario
> **Stakeholder:** Chief Compliance Officer (CCO) & FinCEN Liaison  
> *"Ingest OFAC regulatory sanctions from `aml_sanctions_watchlist.yaml`. Scan `customers.csv`, `api_event_logs.json`, and `raw_transactions.csv`. Build a comprehensive multi-vector asset freeze docket flagging any customer meeting ANY of the following 3 criteria:
> 1. Country of registration is an OFAC sanctioned nation.
> 2. Login IP address originates from a sanctioned subnet prefix.
> 3. Completed transaction volume with an associated `is_pep == 1` flag > $50,000.
> Generate an executive FinCEN docket detailing flagged customers, frozen capital, and violation classification."*

In [ ]:
# ✍️ PANDAS SOLUTION FOR CASE 8:

In [ ]:
%%sql
-- ✍️ SQL SOLUTION FOR CASE 8: